<a href="https://colab.research.google.com/github/sanjana19991118/engineering-mindmaps-Knowledge-base/blob/main/Advanced_Apex_Project_2_Week2_Sanjana_V_12000191_skin_cancer_cnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CNN-Based Medical Image Classification for Disease Detection
### Dataset: HAM10000 Skin Cancer MNIST Dataset

This notebook implements the complete step-by-step pipeline for automated skin cancer classification from dermoscopic images, incorporating data leakage mitigation and class imbalance handling.

## Step 1: Data Loading

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from sklearn.model_selection import GroupKFold
from sklearn.utils.class_weight import compute_class_weight

# Load the tabular metadata
metadata_path = 'HAM10000_metadata.csv'
df = pd.read_csv(metadata_path)
print(f"Dataset loaded successfully. Total rows: {len(df)}")
df.head()

## Step 2: Data Preprocessing

In [ ]:
# Handle missing values in the age column using median imputation
median_age = df['age'].median()
df['age'].fillna(median_age, inplace=True)

# Standardize categorical descriptions labeled as 'unknown'
df['sex'] = df['sex'].replace('unknown', 'Unknown')
df['localization'] = df['localization'].replace('unknown', 'Unknown')

print("Missing values and unknowns handled successfully.")
df.isnull().sum()

## Step 3: Exploratory Data Analysis (EDA)

In [ ]:
# Visualize the 67% severe class imbalance
plt.figure(figsize=(10, 5))
sns.countplot(data=df, x='dx', order=df['dx'].value_counts().index, palette='viridis')
plt.title('Distribution of Diagnostic Categories (Target Variable: dx)')
plt.xlabel('Lesion Type')
plt.ylabel('Count')
plt.show()

# Calculate explicit distribution proportions
print(df['dx'].value_counts(normalize=True) * 100)

## Step 4: Feature Engineering & Splitting (Leakage Prevention)

In [ ]:
# Prevent data leakage by enforcing GroupKFold partitioning grouped by lesion_id
gkf = GroupKFold(n_splits=5)
df['fold'] = -1

for fold, (train_idx, val_idx) in enumerate(gkf.split(df, groups=df['lesion_id'])):
    df.loc[val_idx, 'fold'] = fold

print("GroupKFold splits successfully calculated. Data leakage via duplicate patient lesions blocked.")
df['fold'].value_counts()

## Step 5: Baseline CNN Model Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms

# Compute class weights to counter the severe imbalance in the loss function
classes = np.unique(df['dx'])
class_weights = compute_class_weight(class_weight='balanced', classes=classes, y=df['dx'])
class_weights_tensor = torch.FloatTensor(class_weights)

# Define a basic custom CNN Architecture baseline floor
class BaselineCNN(nn.Module):
    def __init__(self, num_classes=7):
        super(BaselineCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 56 * 56, 128), # Assuming input image size 224x224
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = BaselineCNN(num_classes=7)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = optim.Adam(model.parameters(), lr=0.001)
print("Baseline model and class-weighted loss criteria defined successfully.")